# Antimeridian (dateline) seam test

Goal: build a CrocoDash case whose domain straddles the 180° seam and see whether grid creation, forcing configuration, and GLORYS extraction handle the wraparound correctly.

Note: the originally requested `center_lon=160` does **not** actually cross 180° — with `width_m=2,000,000`, `height_m=3,000,000` at `center_lat=-50`, that grid spans only ~140.8–179.2°E. Checked empirically:

```
center_lon=160 -> tlon min/max: 140.78, 179.22   (no crossing)
center_lon=170 -> tlon min/max: -179.999, 179.999 (crosses)
center_lon=175 -> tlon min/max: -179.999, 179.999 (crosses)
```

So this notebook uses `center_lon=170` (same width/height/resolution otherwise) to actually exercise the seam.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from mom6_forge.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid
from CrocoDash.case import Case

In [ ]:
# CESM case (experiment) name
casename = "seam.test.1"

# CESM source root (CROCESM_workshop_2025 is cime6.1.118 -- CrocoDash's cime_interface
# now requires cime minor version 2, so use full_regional_cesm (cime6.2.2) instead)
cesmroot = Path("~/work/installs/full_regional_cesm").expanduser()

# Place where all input files go
inputdir = Path.home() / "scratch" / "croc_input" / casename

# CESM case directory
caseroot = Path.home() / "croc_cases" / casename

compset = "1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_SWAV"

## Grid: `Grid.from_center` across the dateline

`center_lon=170` (shifted from the requested 160 to actually straddle 180° — see note above).

In [ ]:
grid2 = Grid.from_center(
    center_lon=170,
    center_lat=-50,
    width_m=2000000,
    height_m=3000000,
    resolution_m=10000,
    name="seam_test",
)

print("tlon min/max:", float(grid2.tlon.min()), float(grid2.tlon.max()))
print("tlat min/max:", float(grid2.tlat.min()), float(grid2.tlat.max()))
print("shape:", grid2.tlon.shape)

In [ ]:
# Flat bathymetry + uniform vertical grid — fast/minimal setup, not physically tuned
topo = Topo(grid2, min_depth=10.0, git=False)
topo.set_flat(1000.0)

vgrid = VGrid.uniform(nk=10, depth=1000.0, name="seam_test")

## Case setup

In [ ]:
case = Case(
    cesmroot=cesmroot,
    caseroot=caseroot,
    inputdir=inputdir,
    compset=compset,
    ocn_grid=grid2,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    override=True,
)

In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    function_name="get_glorys_data_from_rda",
)

In [ ]:
# This is the step that actually exercises the seam: GLORYS bounding-box computation
# and dateline-wraparound download/regrid logic (CrocoDash/raw_data_access/datasets/glorys.py).
# Can take several minutes to tens of minutes.
case.process_forcings()

## Result: found + fixed a real bug

The first run of `process_forcings()` above (with unpatched `CrocoDash/raw_data_access/datasets/glorys.py`)
**crashed** while regridding a boundary. On inspection, `time` on the raw fetched GLORYS file was corrupted.

The fix is in an already-open, unmerged PR: **[CROCODILE-CESM/CrocoDash#230](https://github.com/CROCODILE-CESM/CrocoDash/pull/230)**
("GLORYS change longitude slicer", base branch `dask_extract_forcing`, fixes #142). It deletes the
hand-rolled `convert_lons_to_180_range` + `xr.concat` two-slice + `sortby` dateline logic in
`get_glorys_data_from_rda` entirely, replacing it with the existing, more robust
`mom6_forge.utils.longitude_slicer` helper:

```python
dataset = ds.sel(latitude=slice(lat_min - 1, lat_max + 1))
buf = 1.0
dataset = longitude_slicer(dataset, [lon_min - buf, lon_max + buf], longitude_coords="longitude")
```

Applying that diff to `glorys.py` and clearing the (stale, partially-corrupted) `raw_data`/`regridded_data`
caches, `case.process_forcings()` (equivalently `driver.py --all` in `extract_forcings/`) completed cleanly
end-to-end in ~5 minutes: all 4 OBC segments + initial condition regridded and merged into `ocnice/`, e.g.

```
lon_segment_003 (west): 150.66 .. 159.01           # normal side
lon_segment_004 (east): -179.01 .. -170.66          # wrapped side, no discontinuity/NaNs
```

**Caveat on the very first crash**: one of the four raw boundary files (`south_unprocessed...nc`) turned out
to be a leftover partial write from an earlier *interrupted* run (killed via `qdel`/`kill -9` while
mid-download) — its `time` variable was all-NaN due to that interruption, not the seam-crossing code path
itself. `north`/`west`/`east`, freshly downloaded in one uninterrupted run, had valid monotonic time even
with the *old* buggy code. So the antimeridian-longitude-handling rewrite in PR #230 is confirmed necessary
and sufficient to get a clean run, but the very first traceback we hit was partly an artifact of process
management, not proof the old code corrupts every seam-crossing fetch's time coordinate.

**To reproduce this fix locally** (until #230 merges): add `from mom6_forge.utils import longitude_slicer`
to `glorys.py`, replace the `lon_min, lon_max = convert_lons_to_180_range(...)` + `if lon_min*lon_max>0: ... else: ...`
block with the two lines above, then delete any previously-fetched `extract_forcings/raw_data/*_unprocessed*.nc`
and `extract_forcings/regridded_data/*` before rerunning.

## Sanity checks after processing

Confirm the boundary/IC forcing files were actually written and their longitude coordinate spans the seam correctly (no empty slices, no discontinuity).

In [ ]:
import xarray as xr

ocnice_dir = case.inputdir / "ocnice"
for f in sorted(ocnice_dir.glob("*")):
    print(f.name)

# Check the east/west boundary longitude ranges straddle 180 cleanly (no NaNs, no discontinuity)
for seg, label in [("segment_003", "west"), ("segment_004", "east")]:
    ds = xr.open_dataset(ocnice_dir / f"forcing_obc_{seg}.nc")
    lon_var = f"lon_{seg}"
    print(label, float(ds[lon_var].min()), float(ds[lon_var].max()))

## Part 2: does the case actually *run* in MOM6? — found a second, deeper bug

Building and submitting this case on Derecho (`case.setup` → `case.build` → `case.submit`) crashed on the
very first timestep:

```
FATAL from PE 9: NaN in input field of reproducing_EFP_sum(_2d).
```

always on the same MPI ranks. Reducing `DT`/`DT_THERM` (600s/1800s → 300s/900s) did **not** help — the
identical crash on the identical ranks — which ruled out ordinary CFL/numerical instability and pointed at
something rank/tile-specific.

**Root cause:** `mom6_forge/_supergrid.py`'s `ProjectedSupergrid.from_center`/`from_crs` (used by
`Grid.from_center`) call `_init_from_xy(...)` without specifying `dx_dy_calc_type`, so they silently fall
back to the default `"smallangle"` method:

```python
dx = R * np.cos(np.deg2rad(y[:, :-1])) * np.deg2rad(np.diff(x, axis=1))
```

`np.diff(x, axis=1)` is a **raw** longitude difference with no antimeridian unwrapping. At the seam, `x`
jumps from `+179.96°` to `-179.98°`, giving `np.diff = -359.77`° instead of the true `+0.23°` — producing a
single phantom grid cell with `dx ≈ -32,384 km` (one full latitude-circle circumference) instead of the
correct ~5 km, *at every row* of the grid. This directly contradicts the class's own docstring, which
claims dx/dy/area/angle_dx are "computed from exact great-circle geometry" — that's only actually true for
`dx_dy_calc_type="haversine"` (uses `sin(dlon/2)**2`, safe against the ±360° branch cut because squaring
removes the sign ambiguity), not the silently-defaulted `"smallangle"` path.

**Fix:** pass `dx_dy_calc_type="haversine"` explicitly in both `from_center` and `from_crs`. Applied to all
three copies in this checkout: `regional_mom_workflows/mom6_forge/mom6_forge/_supergrid.py` (the one
actually resolved by `import mom6_forge` in the `CrocoDash` conda env), plus the two embedded copies under
`CrocoDash/CrocoDash/visualCaseGen/external/mom6_forge/` (main checkout and the
`manish/interior-obc-segment` worktree). **Not yet upstreamed as a PR.**

Verified: `dx` is now uniform ~4935-4985m across the whole grid, no negative values. Regenerated the case,
forcings, and rebuilt/resubmitted on Derecho (small PE layout: 10 tasks/component, rootpe 0, `develop`
queue, 20 min walltime) — **the model ran cleanly for 7 full simulated days (2020/01/01 → 2020/01/08)**
crossing the antimeridian, with sane energy/mass/salt/heat conservation (errors ~1e-4 to 1e-3) and
`MaxCFL` staying under 0.3 throughout. It then stopped for an unrelated, trivial reason — the OBC forcing
time axis ends exactly at the run's last needed timestamp minus 10 minutes (a `date_range`/`STOP_N`
mismatch, easily fixed by widening `date_range` by a day) — not a bug.

**This `dx_dy_calc_type` bug affects any `Grid.from_center`/`from_crs` domain crossing ±180° longitude**,
independent of the CrocoDash/GLORYS fix above. It may also be worth re-checking the polar domain test cases
(`EPSG:3995`/`3031`, Arctic/Antarctic) for the same issue, since a pole-centered grid can cross the seam
internally depending on orientation.